# Mini-Challenge 3 — Clean a messy export
### Saint Mary's · Thursday afternoon, 12:25

> *"This file came in from the radiology system. Different encoding, different separator, weird missing values. Make it usable."*

**Time:** 15 min · **Mode:** solo · **Goal:** see the most common loading bugs and fix them.

---

This is exactly what you receive in real life. Welcome to data engineering.

# Solution — Mini-Challenge 3

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Walk up to find the 'data/' folder — works from any notebook location
HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found"
print("Using DATA =", DATA)

Using DATA = c:\Obsidian\Arbeitsraum Claude\03_Lehre_&_Forschung\Python_Lehrnotizen\Kurs-Repo\data


In [16]:
df = pd.read_csv(
    DATA / "messy_export.csv",
    encoding="cp1252",
    sep=";",
    decimal=",",
    na_values=["", "n/a", "N/A"],
)
print(f"Loaded {len(df)} rows")
df
df.info()
df.describe()

Loaded 9 rows
<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Patient ID  9 non-null      str    
 1   Age         7 non-null      float64
 2   Gender      9 non-null      str    
 3   Insurance   9 non-null      str    
 4   LOS         9 non-null      int64  
 5   Cost (EUR)  7 non-null      float64
dtypes: float64(2), int64(1), str(3)
memory usage: 564.0 bytes


,Age,LOS,Cost (EUR)
count,7.000000,9.000000,7.000000
mean,66.142857,4.111111,3590.071429
std,11.378760,2.891559,2678.598210
min,48.000000,-1.000000,1290.000000
25%,61.000000,3.000000,1635.000000
50%,67.000000,4.000000,3450.000000
75%,72.000000,5.000000,4085.250000
max,82.000000,9.000000,8950.000000


## Clean column names

In [12]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("(eur)", "eur", regex=False).str.strip("_")
print(df.columns.tolist())

['patient_id', 'age', 'gender', 'insurance', 'los', 'cost_eur']


## Drop duplicates and impossible values

In [13]:
before = len(df)
df = df.drop_duplicates()
df = df[df["los"].notna() & (df["los"] >= 0)]
print(f"Removed {before - len(df)} rows. Now {len(df)} rows.")
df

Removed 2 rows. Now 7 rows.


,patient_id,age,gender,insurance,los,cost_eur
0,P0001,67.0,F,AOK,5,3450.0
1,P0002,NaN,M,TK,3,1820.0
2,P0003,82.0,female,Barmer,4,NaN
4,P0005,71.0,M,Privat - Allianz,7,4720.5
6,P0006,NaN,M,AOK,2,1290.0
7,P0007,48.0,F,TK,3,NaN
8,P0008,73.0,M,Selbstzahler,9,8950.0


## DD — normalise gender

In [14]:
df["gender"] = df["gender"].str.upper().str[0]
df["patient_id"] = df["patient_id"].str.strip()
df

,patient_id,age,gender,insurance,los,cost_eur
0,P0001,67.0,F,AOK,5,3450.0
1,P0002,NaN,M,TK,3,1820.0
2,P0003,82.0,F,Barmer,4,NaN
4,P0005,71.0,M,Privat - Allianz,7,4720.5
6,P0006,NaN,M,AOK,2,1290.0
7,P0007,48.0,F,TK,3,NaN
8,P0008,73.0,M,Selbstzahler,9,8950.0


**Why does `.str.upper().str[0]` work?**

`'female'.upper() = 'FEMALE'`, then `[0] = 'F'`. `'F'.upper() = 'F'`, then `[0] = 'F'`. Both arrive at `'F'`.
This is the kind of one-liner that earns you trust in a real data team.